# finetuned gemma_vla evaluation

이 노트북은 `finetuning_vla.py`로 학습된 `gemma_vla` 체크포인트를 불러와, 몇 개 샘플에 대해 ground-truth answer와 모델이 생성한 answer를 나란히 비교합니다.

아래의 경로만 본인 환경에 맞게 수정한 뒤 순서대로 실행하면 됩니다.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
import sys
sys.path.append("/zfsauton2/home/mineuih/waymax_rs")

from data.utils import split_cache_paths, resolve_cache_paths
from train_vla.utils.utils import resolve_device, resolve_dtype, move_features_to_device, tokenize_text_batch
from model.vla.gemma_vla_v2 import VecSceneGemmaVLA

# --- paths / sampling settings ---
RUN_DIR = Path("/zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_without_subgoal_20260620_105846")
CHECKPOINT_PATH = RUN_DIR / "checkpoints"
CACHE_DIR = Path("/zfsauton/scratch/mineuih/waymax_rs/sim_state_cache_npz")
ANNOTATION_DIR = Path("/zfsauton/scratch/mineuih/waymax_rs/annotations")
FILE_INDICES = None  # 예: [0, 1] 처럼 일부 shard만 보고 싶으면 지정
ANCHOR_STEPS = (10,)
MAX_SAMPLES = 8
BATCH_SIZE = 1

MAX_PROMPT_LENGTH = 128
MAX_ANSWER_LENGTH = 64
ADD_EOS = False

# training_config.json이 저장된 run_dir를 기준으로 모델 하이퍼파라미터를 복원합니다.
MODEL_TYPE = "gemma"
GEMMA_NAME = "google/gemma-4-E2B-it"


ModuleNotFoundError: No module named 'data'

In [ ]:
import json
import sys
from typing import Any

repo_root = Path("/zfsauton2/home/mineuih/waymax_rs")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


def resolve_latest_checkpoint(path: Path) -> Path:
    path = path.expanduser().resolve()
    if path.is_file():
        return path
    if path.is_dir():
        checkpoint_dir = path if path.name == "checkpoints" else path / "checkpoints"
        checkpoint_files = sorted(checkpoint_dir.glob("step_*.pt"))
        if not checkpoint_files:
            raise FileNotFoundError(f"No checkpoint files found under {checkpoint_dir}")
        return checkpoint_files[-1]
    raise FileNotFoundError(f"Checkpoint path not found: {path}")


def load_training_config(run_dir: Path) -> dict[str, Any]:
    config_path = run_dir / "training_config.json"
    if not config_path.exists():
        raise FileNotFoundError(f"training_config.json not found: {config_path}")
    with config_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def normalize_text(text: str) -> str:
    return " ".join(text.strip().lower().split())


checkpoint_path = resolve_latest_checkpoint(CHECKPOINT_PATH)
run_dir = checkpoint_path.parent.parent if checkpoint_path.parent.name == "checkpoints" else checkpoint_path.parent
run_cfg = load_training_config(run_dir)

# model / device
# from_pretrained가 training_config.json을 읽어서 구조를 복원합니다.
device = resolve_device()
dtype = resolve_dtype(run_cfg.get("dtype", "bf16"))
model = VecSceneGemmaVLA.from_pretrained(checkpoint_path, map_location="cpu", device=device, dtype=dtype)
model.eval()

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Run dir: {run_dir}")
print(f"Device: {device}, dtype: {dtype}")
print(f"Model type: {run_cfg.get('model_type', MODEL_TYPE)}")


In [ ]:
from data.cache_loader import build_dataloader
from data.types import PreprocessConfig

preprocess_dict = run_cfg.get("preprocess_cfg", {})
preprocess_cfg = PreprocessConfig(**preprocess_dict) if isinstance(preprocess_dict, dict) else preprocess_dict

cache_paths = resolve_cache_paths(str(CACHE_DIR), FILE_INDICES)
train_cache_paths, val_cache_paths = split_cache_paths(cache_paths, run_cfg.get("validation_fraction", 0.2))
eval_cache_paths = val_cache_paths or train_cache_paths

loader = build_dataloader(
    str(CACHE_DIR),
    preprocess_cfg=preprocess_cfg,
    annotation_dir=str(ANNOTATION_DIR),
    anchor_steps=ANCHOR_STEPS,
    language_label="instruction",
    backend="torch",
    include_inst_features=False,
    file_indices=FILE_INDICES,
    batch_size=BATCH_SIZE,
    shuffle_seed=0,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    cache_paths=eval_cache_paths,
)

rows: list[dict[str, Any]] = []
seen = 0

for batch in loader:
    prompts = list(batch.prompts or [])
    answers = list(batch.answers or [])
    if not prompts:
        continue

    remaining = MAX_SAMPLES - seen
    if remaining <= 0:
        break
    if len(prompts) > remaining:
        prompts = prompts[:remaining]
        answers = answers[:remaining]
        batch_features = {key: value[:remaining] for key, value in batch.features.items()}
    else:
        batch_features = batch.features

    prompt_ids, prompt_mask, _, _ = tokenize_text_batch(
        model.tokenizer,
        prompts,
        answers,
        add_eos=ADD_EOS,
        device=device,
        max_prompt_length=MAX_PROMPT_LENGTH,
        max_answer_length=MAX_ANSWER_LENGTH,
    )
    batch_features = move_features_to_device(batch_features, device, dtype)

    amp_enabled = device.type == "cuda"
    with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=dtype, enabled=amp_enabled):
        predictions = model.generate_predictions(
            input_features=batch_features,
            prompt_ids=prompt_ids,
            prompt_mask=prompt_mask,
            max_new_tokens=MAX_ANSWER_LENGTH,
        )

    predicted_answers = predictions["answer"]
    subgoal_pred = predictions.get("subgoal_pred")
    subgoal_gt = None
    if "subgoal_xy" in batch_features:
        subgoal_gt = (100.0 * batch_features["subgoal_xy"].detach().float().cpu().numpy()).tolist()

    for idx, (prompt, gt, pred) in enumerate(zip(prompts, answers, predicted_answers)):
        row: dict[str, Any] = {
            "sample_id": seen + idx,
            "prompt": prompt,
            "ground_truth": gt,
            "prediction": pred,
            "exact_match": normalize_text(gt) == normalize_text(pred),
            "match_output": gt.strip().lower() == pred.strip().lower(),
        }
        if subgoal_pred is not None:
            row["subgoal_pred"] = subgoal_pred[idx].tolist() if hasattr(subgoal_pred[idx], "tolist") else subgoal_pred[idx]
        if subgoal_gt is not None:
            row["subgoal_gt"] = subgoal_gt[idx]
        rows.append(row)

    seen += len(prompts)
    if seen >= MAX_SAMPLES:
        break

df = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 120, "display.width", 160):
    display(df)

print(f"exact_match mean: {df['exact_match'].mean() if not df.empty else 0.0:.3f}")
print(f"match_output mean: {df['match_output'].mean() if not df.empty else 0.0:.3f}")
